In [53]:
%pip install -q -U transformers huggingface_hub timm accelerate safetensors

Note: you may need to restart the kernel to use updated packages.


In [54]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# Get token from Kaggle Secrets
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

# Log in
login(token=hf_token)

print("Hugging Face login successful!")

Hugging Face login successful!


In [55]:
from huggingface_hub import snapshot_download

MODEL_ID = "facebook/dinov3-vitl16-pretrain-lvd1689m"

model_path = snapshot_download(
    repo_id=MODEL_ID,
    token=hf_token
)

print("Model downloaded to:", model_path)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Model downloaded to: /root/.cache/huggingface/hub/models--facebook--dinov3-vitl16-pretrain-lvd1689m/snapshots/ea8dc2863c51be0a264bab82070e3e8836b02d51


In [56]:
import os, gc, zipfile, warnings, random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from transformers import AutoModel

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

warnings.filterwarnings('ignore')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if torch.cuda.is_available():
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

Device: cuda
VRAM: 15.6 GB


In [57]:
CFG = dict(
    # ── Model ──────────────────────────────────────────────────────
    # vitsplus is wider than vits — same depth, more channels → better features
    model_name = 'facebook/dinov3-vits16plus-pretrain-lvd1689m',
    img_size     = 384,      # higher res → better fine-grained animal features

    # ── CV / Training ───────────────────────────────────────────────
    n_folds      = 5,
    seed         = 42,
    epochs       = 20,
    batch_size   = 16,       # 384px needs lower bs
    grad_accum   = 4,        # effective batch = 64
    freeze_epochs= 2,        # freeze first 4 backbone blocks for N epochs

    # ── LR ─────────────────────────────────────────────────────────
    backbone_lr  = 5e-5,     # higher than before — vitsplus can handle it
    head_lr      = 5e-4,
    weight_decay = 1e-4,
    warmup_frac  = 0.05,

    # ── Regularisation ──────────────────────────────────────────────
    label_smooth = 0.1,
    mixup_alpha  = 0.4,
    cutmix_alpha = 1.0,
    mix_prob     = 0.5,      # prob of applying mixup/cutmix each batch
    erasing_prob = 0.25,

    mixed_prec   = True,
    num_workers  = 4,
    output_dir   = '/kaggle/working',
)
os.makedirs(CFG['output_dir'], exist_ok=True)

def seed_everything(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_everything(CFG['seed'])

In [58]:
# ─── Data ─────────────────────────────────────────────────────────────────────
def find_file(name):
    for r, _, files in os.walk('/kaggle/input'):
        if name in files: return os.path.join(r, name)
    raise FileNotFoundError(name)

train_df = pd.read_csv(find_file('train.csv'))
sub_df   = pd.read_csv(find_file('sample_submission.csv'))
base     = os.path.dirname(find_file('train.csv'))
TRAIN_DIR, TEST_DIR = base+'/train_images', base+'/test_images'

classes           = sorted(train_df['animal_class'].unique())
class_to_idx      = {c: i for i, c in enumerate(classes)}
idx_to_class      = {i: c for c, i in class_to_idx.items()}
train_df['label'] = train_df['animal_class'].map(class_to_idx)
CFG['num_classes'] = len(classes)
print(f'Train {len(train_df)} | Test {len(sub_df)} | Classes {CFG["num_classes"]}')

Train 3087 | Test 1179 | Classes 72


In [59]:
# ─── Transforms ───────────────────────────────────────────────────────────────
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
S = CFG['img_size']

train_tfm = T.Compose([
    T.Resize((S+48, S+48), interpolation=T.InterpolationMode.BICUBIC),
    T.RandomCrop(S),
    T.RandomHorizontalFlip(0.5),
    T.RandAugment(num_ops=2, magnitude=9),   # stronger than ColorJitter alone
    T.ToTensor(),
    T.Normalize(MEAN, STD),
    T.RandomErasing(p=CFG['erasing_prob'], scale=(0.02, 0.2)),
])
val_tfm = T.Compose([
    T.Resize((S, S), interpolation=T.InterpolationMode.BICUBIC),
    T.ToTensor(), T.Normalize(MEAN, STD),
])
tta_tfm = T.Compose([
    T.Resize((S, S), interpolation=T.InterpolationMode.BICUBIC),
    T.RandomHorizontalFlip(p=1.0),
    T.ToTensor(), T.Normalize(MEAN, STD),
])

In [60]:
# ─── Dataset ──────────────────────────────────────────────────────────────────
class AnimalDataset(Dataset):
    def __init__(self, df, image_dir, transform, is_test=False):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform
        self.is_test = is_test

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.image_dir, row['image_id'])).convert('RGB')
        img = self.transform(img)
        if self.is_test: return img, row['image_id']
        return img, torch.tensor(row['label'], dtype=torch.long)

In [61]:
# ─── Mixup / CutMix ───────────────────────────────────────────────────────────
def rand_bbox(size, lam):
    W, H = size[2], size[3]
    cut_rat = np.sqrt(1. - lam)
    cut_w, cut_h = int(W * cut_rat), int(H * cut_rat)
    cx, cy = np.random.randint(W), np.random.randint(H)
    x1 = np.clip(cx - cut_w // 2, 0, W)
    y1 = np.clip(cy - cut_h // 2, 0, H)
    x2 = np.clip(cx + cut_w // 2, 0, W)
    y2 = np.clip(cy + cut_h // 2, 0, H)
    return x1, y1, x2, y2

def mixup_cutmix(x, y, num_classes):
    """Returns mixed (x, y_a, y_b, lam). Apply only during training."""
    if random.random() > CFG['mix_prob']:
        return x, y, y, 1.0

    use_cutmix = random.random() > 0.5
    if use_cutmix:
        lam = np.random.beta(CFG['cutmix_alpha'], CFG['cutmix_alpha'])
        idx = torch.randperm(x.size(0), device=x.device)
        x1, y1, x2, y2 = rand_bbox(x.size(), lam)
        x[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]
        lam = 1 - (x2-x1)*(y2-y1) / (x.size(2)*x.size(3))
    else:
        lam = np.random.beta(CFG['mixup_alpha'], CFG['mixup_alpha'])
        idx = torch.randperm(x.size(0), device=x.device)
        x = lam * x + (1 - lam) * x[idx]

    return x, y, y[idx], lam

def mix_criterion(logits, y_a, y_b, lam):
    ce = nn.CrossEntropyLoss(label_smoothing=CFG['label_smooth'])
    return lam * ce(logits, y_a) + (1 - lam) * ce(logits, y_b)

In [62]:
# ─── Model ────────────────────────────────────────────────────────────────────
class DINOv3Classifier(nn.Module):
    def __init__(self, model_name, num_classes, drop=0.3):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        dim = self.backbone.config.hidden_size
        self.head = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Dropout(drop),
            nn.Linear(dim, num_classes),
        )

    def forward(self, x):
        return self.head(self.backbone(pixel_values=x).pooler_output)

    def freeze_blocks(self, n):
        """Freeze the first n transformer blocks in Hugging Face DINOv3."""
        backbone = self.backbone

        # DINOv3ViTModel exposes its transformer through backbone.model
        core = getattr(backbone, "model", None)
        if core is None:
            raise AttributeError(
                f"Cannot find backbone.model. Children: "
                f"{list(dict(backbone.named_children()).keys())}"
            )

        # Find the transformer block container
        layers = None
        for name in ("layer", "layers", "blocks"):
            candidate = getattr(core, name, None)
            if candidate is not None:
                try:
                    if len(candidate) > 0:
                        layers = candidate
                        print(f"Found {len(layers)} blocks at backbone.model.{name}")
                        break
                except TypeError:
                    pass

        if layers is None:
            print("Core modules:", list(dict(core.named_children()).keys()))
            raise AttributeError("Could not locate DINOv3 transformer blocks inside backbone.model")

        # Freeze the first n blocks
        for block in list(layers)[:n]:
            for param in block.parameters():
                param.requires_grad = False

    def unfreeze_all(self):
        for p in self.backbone.parameters():
            p.requires_grad = True

def build_model():
    m = DINOv3Classifier(CFG['model_name'], CFG['num_classes'])
    return m.to(device)

m = build_model()
print(f'Params: {sum(p.numel() for p in m.parameters())/1e6:.1f}M  '
      f'| hidden: {m.backbone.config.hidden_size}')
del m; gc.collect(); torch.cuda.empty_cache()

Loading weights:   0%|          | 0/235 [00:00<?, ?it/s]

Params: 28.7M  | hidden: 384


In [63]:
# ─── Optimizer + cosine scheduler ─────────────────────────────────────────────
def make_opt_sched(model, total_steps):
    head_ids = {id(p) for p in model.head.parameters()}
    opt = torch.optim.AdamW([
        {'params': [p for p in model.parameters() if id(p) not in head_ids],
         'lr': CFG['backbone_lr']},
        {'params': list(model.head.parameters()),
         'lr': CFG['head_lr']},
    ], weight_decay=CFG['weight_decay'])
    warmup = int(total_steps * CFG['warmup_frac'])
    def lr_fn(s):
        if s < warmup: return s / max(1, warmup)
        p = (s - warmup) / max(1, total_steps - warmup)
        return max(0.0, 0.5 * (1 + np.cos(np.pi * p)))
    return opt, torch.optim.lr_scheduler.LambdaLR(opt, lr_fn)

@torch.no_grad()
def validate(model, loader):
    model.eval()
    preds, labels, loss = [], [], 0.0
    ce = nn.CrossEntropyLoss(label_smoothing=CFG['label_smooth'])
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        with torch.cuda.amp.autocast(enabled=CFG['mixed_prec']):
            logits = model(x)
        loss += ce(logits, y).item()
        preds.extend(logits.argmax(1).cpu().numpy())
        labels.extend(y.cpu().numpy())
    return loss/len(loader), f1_score(labels, preds, average='macro', zero_division=0)

In [64]:
# ─── 5-Fold Training ──────────────────────────────────────────────────────────
skf = StratifiedKFold(n_splits=CFG['n_folds'], shuffle=True, random_state=CFG['seed'])
oof_probs, fold_scores = np.zeros((len(train_df), CFG['num_classes'])), []

for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, train_df['label'])):
    print(f'\n{"="*55}\n  FOLD {fold+1}/{CFG["n_folds"]}\n{"="*55}')
    seed_everything(CFG['seed'] + fold)

    tr_dl = DataLoader(AnimalDataset(train_df.iloc[tr_idx], TRAIN_DIR, train_tfm),
                       batch_size=CFG['batch_size'], shuffle=True,
                       num_workers=CFG['num_workers'], pin_memory=True, drop_last=True)
    va_dl = DataLoader(AnimalDataset(train_df.iloc[va_idx], TRAIN_DIR, val_tfm),
                       batch_size=CFG['batch_size'], shuffle=False,
                       num_workers=CFG['num_workers'], pin_memory=True)

    model  = build_model()
    model.freeze_blocks(4)   # freeze first 4 blocks initially

    total_steps = (len(tr_dl) // CFG['grad_accum']) * CFG['epochs']
    opt, sched  = make_opt_sched(model, total_steps)
    scaler      = torch.cuda.amp.GradScaler(enabled=CFG['mixed_prec'])
    best_f1     = 0.0
    ckpt        = os.path.join(CFG['output_dir'], f'fold{fold+1}_best.pt')

    for epoch in range(CFG['epochs']):
        # unfreeze backbone after warmup epochs
        if epoch == CFG['freeze_epochs']:
            model.unfreeze_all()
            print('  [backbone unfrozen]')

        model.train(); opt.zero_grad(); tr_loss = 0.0
        for step, (x, y) in enumerate(tqdm(tr_dl,
                desc=f'  Ep {epoch+1:2d}/{CFG["epochs"]}', leave=False)):
            x, y = x.to(device), y.to(device)
            x, ya, yb, lam = mixup_cutmix(x, y, CFG['num_classes'])
            with torch.cuda.amp.autocast(enabled=CFG['mixed_prec']):
                logits = model(x)
                loss = mix_criterion(logits, ya, yb, lam) / CFG['grad_accum']
            scaler.scale(loss).backward()
            tr_loss += loss.item() * CFG['grad_accum']
            if (step+1) % CFG['grad_accum'] == 0:
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad()

        vl, vf1 = validate(model, va_dl)
        flag = '  ✓' if vf1 > best_f1 else ''
        print(f'  Ep {epoch+1:2d} | tr {tr_loss/len(tr_dl):.4f} '
              f'| val {vl:.4f} | F1 {vf1:.4f}{flag}')
        if vf1 > best_f1:
            best_f1 = vf1
            torch.save({'model_state': model.state_dict(), 'fold': fold+1,
                        'epoch': epoch+1, 'val_macro_f1': best_f1,
                        'classes': classes, 'cfg': CFG}, ckpt)

    print(f'  → best F1: {best_f1:.4f}')
    fold_scores.append(best_f1)

    # OOF
    model.load_state_dict(torch.load(ckpt)['model_state'])
    model.eval(); offset = 0
    with torch.no_grad():
        for x, _ in DataLoader(AnimalDataset(train_df.iloc[va_idx], TRAIN_DIR, val_tfm),
                                batch_size=CFG['batch_size'], num_workers=CFG['num_workers']):
            p = F.softmax(model(x.to(device)), 1).cpu().numpy()
            oof_probs[va_idx[offset:offset+len(p)]] = p; offset += len(p)

    del model, opt, sched, scaler; gc.collect(); torch.cuda.empty_cache()

cv_mean, cv_std = np.mean(fold_scores), np.std(fold_scores)
oof_f1 = f1_score(train_df['label'], oof_probs.argmax(1), average='macro', zero_division=0)
print(f'\nCV: {cv_mean:.4f} ± {cv_std:.4f}  |  OOF F1: {oof_f1:.4f}')


  FOLD 1/5


Loading weights:   0%|          | 0/235 [00:00<?, ?it/s]

Found 12 blocks at backbone.model.layer


  Ep  1/20:   0%|          | 0/154 [00:00<?, ?it/s]

  Ep  1 | tr 4.2544 | val 3.6012 | F1 0.2383  ✓


...

  Ep 19/20:   0%|          | 0/154 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ca5da0b1080>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process


  Ep 19 | tr 1.4385 | val 0.9611 | F1 0.9673


  Ep 20/20:   0%|          | 0/154 [00:00<?, ?it/s]

  Ep 20 | tr 1.3911 | val 0.9612 | F1 0.9673
  → best F1: 0.9690

CV: 0.9654 ± 0.0039  |  OOF F1: 0.9655


In [65]:
# ─── Inference: 5-fold × 2 TTA ────────────────────────────────────────────────
test_probs = np.zeros((len(sub_df), CFG['num_classes']))
for fold in range(CFG['n_folds']):
    model = build_model()
    model.load_state_dict(
        torch.load(os.path.join(CFG['output_dir'], f'fold{fold+1}_best.pt'))['model_state'])
    model.eval()
    fold_p = np.zeros_like(test_probs)
    for tfm in [val_tfm, tta_tfm]:
        dl = DataLoader(AnimalDataset(sub_df, TEST_DIR, tfm, is_test=True),
                        batch_size=CFG['batch_size'], shuffle=False,
                        num_workers=CFG['num_workers'])
        buf = []
        with torch.no_grad():
            for x, _ in tqdm(dl, desc=f'Fold {fold+1}', leave=False):
                with torch.cuda.amp.autocast(enabled=CFG['mixed_prec']):
                    buf.append(F.softmax(model(x.to(device)), 1).cpu().numpy())
        fold_p += np.vstack(buf)
    test_probs += fold_p / 2
    del model; gc.collect(); torch.cuda.empty_cache()

test_probs /= CFG['n_folds']
sub_df['animal_class'] = [idx_to_class[i] for i in test_probs.argmax(1)]
sub_path = os.path.join(CFG['output_dir'], 'submission.csv')
sub_df[['image_id','animal_class']].to_csv(sub_path, index=False)
print('submission.csv saved'); print(sub_df.head())

Loading weights:   0%|          | 0/235 [00:00<?, ?it/s]

Fold 1:   0%|          | 0/74 [00:00<?, ?it/s]

Fold 1:   0%|          | 0/74 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/235 [00:00<?, ?it/s]

Fold 2:   0%|          | 0/74 [00:00<?, ?it/s]

...

Fold 4:   0%|          | 0/74 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/235 [00:00<?, ?it/s]

Fold 5:   0%|          | 0/74 [00:00<?, ?it/s]

Fold 5:   0%|          | 0/74 [00:00<?, ?it/s]

submission.csv saved
                               image_id animal_class
0          6eab26f8b156fda90e1bc819.png   woodpecker
1  483f71af1c174ab2a0c31a836267227d.png     antelope
2  563878772a3f47ce98cf9c10039a0219.png   chimpanzee
3  9aa782974228482390c1a237408a967a.png      hamster
4  3ed553219cf049b8bd9a65c300b02fb0.png          cat


In [66]:
# ─── Zip ──────────────────────────────────────────────────────────────────────
zip_path = os.path.join(CFG['output_dir'], 'dinov3_strong_weights.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fold in range(CFG['n_folds']):
        pt = os.path.join(CFG['output_dir'], f'fold{fold+1}_best.pt')
        zf.write(pt, arcname=f'fold{fold+1}_best.pt')
        print(f'  + fold{fold+1}_best.pt  ({os.path.getsize(pt)/1e6:.0f} MB)')
    zf.write(sub_path, arcname='submission.csv')
print(f'\n✅ Done  |  CV {cv_mean:.4f} ± {cv_std:.4f}  |  OOF {oof_f1:.4f}')

  + fold1_best.pt  (115 MB)
  + fold2_best.pt  (115 MB)
  + fold3_best.pt  (115 MB)
  + fold4_best.pt  (115 MB)
  + fold5_best.pt  (115 MB)

✅ Done  |  CV 0.9654 ± 0.0039  |  OOF 0.9655
